In [ ]:
# ============================================================================
# HAB Toxin Production - ML Classification Analysis
# Protocol: 5-Fold × 10-Repeat Stratified CV
#
# Hard-penalty gap diagnostic for Random Forest and XGBoost only:
#   Gap measured  : train accuracy − test accuracy on each outer CV fold
#   Tolerance     : PRIMARY = 10%  |  FALLBACK = 12%
#   Status labels :
#     gap ≤ 10%           → OK
#     10% < gap ≤ 12%     → FLAGGED
#     gap > 12%           → HARD-PENALIZED
#   Behaviour     : all folds kept in aggregated arrays and score lists
#                   (Option B); HARD-PENALIZED folds annotated in both
#                   the penalty CSV and the metrics_summary CSV.
#
# Decision Tree :
#   Structural regularisation only (ccp_alpha, max_depth, min_samples_leaf,
#   etc.).
#
# Generates:
#   (1) Model comparison bar plots  — train & test mean ± 1 SD across 50 folds
#   (2) Confusion matrices          — built from aggregated per-fold test predictions
#   (3) Marginal / uncertainty plots— aggregated per-fold test predicted probabilities
#                                     with 5%–95% per-sample uncertainty bands
#                                     (shaded fill + vertical error bars per dot)
#   (4) Probability density plots   — aggregated per-fold test predicted probabilities
#   (5) Feature importance plots    — intrinsic importance on full dataset (train & test)
#   (6) gap_penalty_log.csv         — per-fold gap status for RF and XGB
#
# CI approach: mean ± 1 SD of per-fold scores (50 folds)
# ============================================================================

import os
os.environ["TF_DETERMINISTIC_OPS"] = "1"
# Run from the notebooks/ directory OR set kernel working directory to HAB_Phase1/
import sys, pathlib


import copy
import gc
import random
import warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
plt.ioff()
import matplotlib.patches as mpatches

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix
)
from sklearn.preprocessing import StandardScaler
import tensorflow as tf
import tensorflow.keras as keras
from scipy.stats import gaussian_kde

warnings.filterwarnings("ignore")

# ============================================================================
# CONFIGURATION  — edit paths / columns / hyperparameters here
# ============================================================================

PROJECT_ROOT = pathlib.Path().resolve()
while PROJECT_ROOT.name != "HAB_Phase1":
    PROJECT_ROOT = PROJECT_ROOT.parent
    if PROJECT_ROOT == PROJECT_ROOT.parent:
        raise RuntimeError("Could not find HAB_Phase1 project root")

sys.path.insert(0, str(PROJECT_ROOT))

from config import DATA_PATH, OUTPUT_DIRS

# DATA_PATH    = "Data/UCDavis_Data_for_Analysis2_220days_ToxinProdMicrocystis.csv"
# OUTPUT_DIR   = "C:/Temp/ML5Models_Analysis_Results"
OUTPUT_DIR = str(OUTPUT_DIRS["04"])  # change for each notebook
DATA_PATH  = str(DATA_PATH)

TARGET_COL   = "ToxinProdc_Microcystis_CC"
SITE_COL     = "Site_Name"
DATE_COL     = "Date"
THRESHOLD    = 4000
RANDOM_STATE = 42

FEATURES = ["Q", "Q_1m", "SpecCond.", "OrgC", "OrgN", "PO4",
            "WaterTemp", "DO", "pH"]

# ANN hyperparameters
ANN_HIDDEN_LAYERS = 3
ANN_ACTIVATION    = "elu"
ANN_NEURONS       = 128
ANN_LEARNING_RATE = 0.0081
ANN_DROPOUT_RATE  = 0.25
ANN_EPOCHS        = 1000
ANN_BATCH_SIZE    = 32
ANN_VAL_SPLIT     = 0.2
ANN_PATIENCE      = 30

# Repeated CV settings
N_SPLITS   = 5
N_REPEATS  = 10
# Total folds = N_SPLITS * N_REPEATS = 50

# Permutation importance repeats (ANN only)
PERM_N_REPEATS = 50

ALL_MODELS = ["Logistic Regression", "Decision Tree", "XGBoost",
              "Random Forest", "ANN"]

DECISION_THR = 0.5

# ============================================================================
# HARD-PENALTY GAP DIAGNOSTIC  (RF and XGB only)
# ============================================================================
# Gap = train_accuracy − test_accuracy on each outer CV fold.
# Status mapping:
#   gap ≤  GAP_PRIMARY              → "OK"
#   GAP_PRIMARY  < gap ≤ GAP_FALLBACK → "FLAGGED"
#   gap > GAP_FALLBACK              → "HARD-PENALIZED"
#
# All folds are kept in aggregated arrays and score lists (Option B).
# HARD-PENALIZED folds are annotated in gap_penalty_log.csv and in
# metrics_summary.csv (extra column: HP_folds_RF / HP_folds_XGB).
# ============================================================================
GAP_PRIMARY  = 0.10   # 10% — boundary between OK and FLAGGED
GAP_FALLBACK = 0.12   # 12% — boundary between FLAGGED and HARD-PENALIZED

PENALIZED_MODELS = {"Random Forest", "XGBoost"}

COLORS = {
    "Logistic Regression": "#9467BD",
    "Decision Tree":       "#8C564B",
    "XGBoost":             "#B85042",
    "Random Forest":       "#1C7293",
    "ANN":                 "#E8A838",
}

BAND_PALETTE = {
    "Logistic Regression": "#C9A8E8",
    "Decision Tree":       "#C4A49E",
    "XGBoost":             "#E0948A",
    "Random Forest":       "#6EB3CC",
    "ANN":                 "#F5D08E",
}

TRAIN_COLOR = "#607D8B"
TEST_COLOR  = "#2E4057"


# ============================================================================
# OUTPUT DIRECTORIES
# ============================================================================
def setup_output_dirs():
    subdirs = [
        "figures/model_comparison",
        "figures/confusion_matrix",
        "figures/uncertainty",
        "figures/density",
        "figures/feature_importance",
        "tables",
        "logs",
    ]
    for sub in subdirs:
        path = os.path.join(OUTPUT_DIR, sub)
        os.makedirs(path, exist_ok=True)
    print("\nOutput directories created successfully")
    print("Output directory:", os.path.abspath(OUTPUT_DIR))


def savefig(fig, name):
    """Save figure as PNG and EPS. Handles Windows / OneDrive path issues."""
    try:
        name = name.replace("\\", "/")
        folder = os.path.abspath(
            os.path.join(OUTPUT_DIR, "figures", os.path.dirname(name))
        )
        os.makedirs(folder, exist_ok=True)
        filename  = os.path.basename(name)
        png_path  = os.path.join(folder, filename + ".png")
        eps_path  = os.path.join(folder, filename + ".eps")
        fig.savefig(png_path, dpi=150, bbox_inches="tight")
        try:
            fig.savefig(eps_path, format="eps", bbox_inches="tight")
        except Exception as e:
            print(f"  EPS save failed for {filename}: {e}")
        print(f"  Saved → {png_path}")
    except Exception as e:
        print(f"\n  ERROR SAVING FIGURE '{name}': {e}")


# ============================================================================
# SEED
# ============================================================================
def set_global_seed(seed: int):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)


# ============================================================================
# HARD-PENALTY GAP HELPER
# ============================================================================
def classify_gap(gap: float) -> tuple:
    """
    Classify an accuracy gap into a status label and whether fallback
    was triggered.

    Returns
    -------
    status           : str   — "OK" | "FLAGGED" | "HARD-PENALIZED"
    fallback_used    : bool  — True when gap exceeded GAP_PRIMARY
    tolerance_used   : float — the tolerance that determined final status
    """
    if gap <= GAP_PRIMARY:
        return "OK", False, GAP_PRIMARY
    elif gap <= GAP_FALLBACK:
        # Exceeded primary (10%) but within fallback (12%) → FLAGGED
        return "FLAGGED", True, GAP_FALLBACK
    else:
        # Exceeded fallback (12%) → HARD-PENALIZED
        return "HARD-PENALIZED", True, GAP_FALLBACK


# ============================================================================
# MODEL BUILDERS  (always return fresh, unfitted instances)
# ============================================================================
def build_lr():
    return LogisticRegression(
        warm_start=False, tol=0.0001, solver="saga", penalty="l1",
        max_iter=1000, fit_intercept=True, class_weight="balanced", C=1,
        random_state=RANDOM_STATE,
    )

def build_dt():
    return DecisionTreeClassifier(
        splitter="random", min_samples_split=5, min_samples_leaf=2,
        min_impurity_decrease=0.0, max_leaf_nodes=None, max_features=0.5,
        max_depth=5, criterion="entropy", class_weight="balanced",
        ccp_alpha=0.01,
    )

def build_xgb():
    return XGBClassifier(
        subsample=0.8, scale_pos_weight=1.0, n_estimators=200,
        min_child_weight=3, max_depth=2, learning_rate=0.05,
        reg_lambda=1.0, gamma=0.3, colsample_bytree=0.5,
        alpha=0.1, colsample_bylevel=0.8278, colsample_bynode=0.7501,
        max_delta_step=0,
        random_state=RANDOM_STATE, verbosity=0, n_jobs=1,
    )

def build_rf():
    return RandomForestClassifier(
        n_estimators=300, max_depth=5, min_samples_split=5,
        min_samples_leaf=8, max_features=0.3, bootstrap=True,
        class_weight="balanced", criterion="gini", ccp_alpha=0.0001,
        min_impurity_decrease=0.01627, min_weight_fraction_leaf=0.01946,
        max_leaf_nodes=57, random_state=RANDOM_STATE,
    )

def build_ann(input_dim: int) -> keras.Model:
    model = keras.Sequential(name="ANN_HAB")
    model.add(keras.layers.Input(shape=(input_dim,)))
    model.add(keras.layers.Dense(ANN_NEURONS, activation=ANN_ACTIVATION))
    model.add(keras.layers.Dropout(ANN_DROPOUT_RATE))
    for _ in range(ANN_HIDDEN_LAYERS - 1):
        model.add(keras.layers.Dense(ANN_NEURONS, activation=ANN_ACTIVATION))
        model.add(keras.layers.Dropout(ANN_DROPOUT_RATE))
    model.add(keras.layers.Dense(1, activation="sigmoid"))
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=ANN_LEARNING_RATE),
        loss="binary_crossentropy",
        metrics=["accuracy"],
    )
    return model


# ============================================================================
# ANN WRAPPER  (sklearn-compatible interface)
# ============================================================================
class ANNWrapper:
    def __init__(self, input_dim):
        self.input_dim = input_dim
        self.model = None

    def fit(self, X, y):
        set_global_seed(RANDOM_STATE)
        self.model = build_ann(self.input_dim)
        cb = keras.callbacks.EarlyStopping(
            patience=ANN_PATIENCE, restore_best_weights=True)
        self.model.fit(
            X, y,
            epochs=ANN_EPOCHS,
            batch_size=ANN_BATCH_SIZE,
            validation_split=ANN_VAL_SPLIT,
            callbacks=[cb], verbose=0,
        )
        self.classes_       = np.array([0, 1])
        self.n_features_in_ = X.shape[1]
        return self

    def predict(self, X):
        probs = self.model.predict(X, verbose=0).ravel()
        return (probs >= DECISION_THR).astype(int)

    def predict_proba(self, X):
        p = np.clip(self.model.predict(X, verbose=0).ravel(), 1e-7, 1 - 1e-7)
        return np.column_stack([1 - p, p])

    def score(self, X, y):
        return accuracy_score(y, self.predict(X))


# ============================================================================
# METRIC HELPER
# ============================================================================
def compute_metrics(y_true, y_pred):
    return {
        "accuracy":  accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall":    recall_score(y_true, y_pred, zero_division=0),
        "f1":        f1_score(y_true, y_pred, zero_division=0),
    }


# ============================================================================
# DATA LOADING
# ============================================================================
def load_data():
    print("Loading data ...")
    df = pd.read_csv(DATA_PATH, parse_dates=[DATE_COL])
    df = df.dropna(subset=FEATURES + [TARGET_COL])
    df["label"] = (df[TARGET_COL] >= THRESHOLD).astype(int)
    X     = df[FEATURES]
    y     = df["label"].values
    dates = df[DATE_COL] if DATE_COL in df.columns else pd.Series(range(len(df)))
    print(f"  Samples: {len(df)}  |  Positive (Risky): {y.sum()}  |  Negative: {(1-y).sum()}")
    return X, y, dates, df


# ============================================================================
# CORE CV LOOP
#
# Hard-penalty gap diagnostic is applied to RF and XGB per fold:
#   gap = train_accuracy − test_accuracy
#   Status stored in penalty_log rows; all fold data kept in aggregations.
#
# Returns per-fold metric lists, aggregated per-fold test arrays, and
# the full penalty_log list of dicts (one row per penalized-model fold).
# ============================================================================
def run_cv(X, y, dates):
    """
    Execute 5-fold x 10-repeat stratified CV for all models.
    Scaling is fitted on each fold's training portion independently.
    """
    rskf = RepeatedStratifiedKFold(
        n_splits=N_SPLITS, n_repeats=N_REPEATS, random_state=RANDOM_STATE
    )
    total_folds = N_SPLITS * N_REPEATS

    train_scores = {m: {k: [] for k in ["accuracy","precision","recall","f1"]}
                    for m in ALL_MODELS}
    test_scores  = {m: {k: [] for k in ["accuracy","precision","recall","f1"]}
                    for m in ALL_MODELS}

    agg_preds = {m: [] for m in ALL_MODELS}
    agg_probs = {m: [] for m in ALL_MODELS}
    agg_true  = {m: [] for m in ALL_MODELS}
    agg_dates = {m: [] for m in ALL_MODELS}

    # Per-sample, per-repeat probability store:
    #   repeat_probs[model][sample_idx] -> list of probabilities (one per repeat)
    # Built during CV; used later to compute 5th/95th percentile per sample.
    n_samples = len(y)
    repeat_probs = {m: [[] for _ in range(n_samples)] for m in ALL_MODELS}

    # Per-fold penalty log rows (RF and XGB only)
    penalty_log = []

    builders = {
        "Logistic Regression": build_lr,
        "Decision Tree":       build_dt,
        "XGBoost":             build_xgb,
        "Random Forest":       build_rf,
    }

    X_np    = X.values if isinstance(X, pd.DataFrame) else X
    dates_a = dates.values if isinstance(dates, pd.Series) else np.array(dates)

    for fold_i, (tr_idx, te_idx) in enumerate(rskf.split(X_np, y)):
        print(f"\n  === Fold {fold_i + 1}/{total_folds} ===")

        scaler   = StandardScaler()
        X_tr_sc  = scaler.fit_transform(X_np[tr_idx])
        X_te_sc  = scaler.transform(X_np[te_idx])
        X_tr_df  = pd.DataFrame(X_tr_sc, columns=FEATURES)
        X_te_df  = pd.DataFrame(X_te_sc, columns=FEATURES)
        y_tr, y_te = y[tr_idx], y[te_idx]

        # ── Sklearn models ────────────────────────────────────────────────────
        for mname, builder in builders.items():
            model = builder()
            model.fit(X_tr_df, y_tr)

            yp_tr = model.predict(X_tr_df)
            tr_metrics = compute_metrics(y_tr, yp_tr)
            for k, v in tr_metrics.items():
                train_scores[mname][k].append(v)

            yp_te  = model.predict(X_te_df)
            ypr_te = model.predict_proba(X_te_df)[:, 1]
            te_metrics = compute_metrics(y_te, yp_te)
            for k, v in te_metrics.items():
                test_scores[mname][k].append(v)

            agg_preds[mname].append(yp_te)
            agg_probs[mname].append(ypr_te)
            agg_true[mname].append(y_te)
            agg_dates[mname].append(dates_a[te_idx])

            # Store per-repeat probability for each test sample
            for local_i, global_i in enumerate(te_idx):
                repeat_probs[mname][global_i].append(float(ypr_te[local_i]))

            # ── Hard-penalty gap diagnostic (RF and XGB only) ─────────────────
            if mname in PENALIZED_MODELS:
                tr_acc  = tr_metrics["accuracy"]
                te_acc  = te_metrics["accuracy"]
                gap     = tr_acc - te_acc
                status, fallback_used, tol_used = classify_gap(gap)

                if status != "OK":
                    console_tag = "*** WARNING ***" if status == "HARD-PENALIZED" \
                                  else "  [FLAGGED]"
                    print(f"    {console_tag}  {mname} fold {fold_i+1}: "
                          f"acc gap={gap:.4f}  "
                          f"(train={tr_acc:.4f}, test={te_acc:.4f})  "
                          f"status={status}")

                penalty_log.append({
                    "Fold":               fold_i + 1,
                    "Model":              mname,
                    "Train_Accuracy":     round(tr_acc, 6),
                    "Test_Accuracy":      round(te_acc, 6),
                    "Gap":                round(gap, 6),
                    "Raw_Accuracy":       round(te_acc, 6),
                    "Penalized_Accuracy": -999 if status == "HARD-PENALIZED"
                                          else round(te_acc, 6),
                    "Tolerance_Used":     tol_used,
                    "Gap_Status":         status,
                    "Fallback_Triggered": fallback_used,
                })

        # ── ANN ───────────────────────────────────────────────────────────────
        print(f"    ANN fold {fold_i + 1}/{total_folds} ...")
        ann = ANNWrapper(X_tr_sc.shape[1])
        ann.fit(X_tr_sc, y_tr)

        yp_tr_ann = ann.predict(X_tr_sc)
        for k, v in compute_metrics(y_tr, yp_tr_ann).items():
            train_scores["ANN"][k].append(v)

        yp_te_ann  = ann.predict(X_te_sc)
        ypr_te_ann = ann.predict_proba(X_te_sc)[:, 1]
        for k, v in compute_metrics(y_te, yp_te_ann).items():
            test_scores["ANN"][k].append(v)

        agg_preds["ANN"].append(yp_te_ann)
        agg_probs["ANN"].append(ypr_te_ann)
        agg_true["ANN"].append(y_te)
        agg_dates["ANN"].append(dates_a[te_idx])

        # Store per-repeat probability for each ANN test sample
        for local_i, global_i in enumerate(te_idx):
            repeat_probs["ANN"][global_i].append(float(ypr_te_ann[local_i]))

        keras.backend.clear_session()
        gc.collect()

    # ── Concatenate per-fold test arrays in fold order ────────────────────────
    agg_preds_out = {m: np.concatenate(agg_preds[m]) for m in ALL_MODELS}
    agg_probs_out = {m: np.concatenate(agg_probs[m]) for m in ALL_MODELS}
    agg_true_out  = {m: np.concatenate(agg_true[m])  for m in ALL_MODELS}
    agg_dates_out = {m: np.concatenate(agg_dates[m]) for m in ALL_MODELS}

    return (train_scores, test_scores,
            agg_preds_out, agg_probs_out, agg_true_out, agg_dates_out,
            penalty_log, repeat_probs)


# ============================================================================
# SAVE PENALTY LOG CSV
# ============================================================================
def save_penalty_log(penalty_log: list):
    """Save per-fold gap diagnostic rows for RF and XGB to a CSV."""
    path = os.path.join(OUTPUT_DIR, "tables", "gap_penalty_log.csv")
    df   = pd.DataFrame(penalty_log)
    # Ensure consistent column order
    cols = ["Fold", "Model", "Train_Accuracy", "Test_Accuracy",
            "Gap", "Raw_Accuracy", "Penalized_Accuracy",
            "Tolerance_Used", "Gap_Status", "Fallback_Triggered"]
    df   = df[cols]
    df.to_csv(path, index=False)
    print(f"  Saved → {path}")

    # Print summary counts
    for mname in PENALIZED_MODELS:
        sub = df[df["Model"] == mname]
        counts = sub["Gap_Status"].value_counts().to_dict()
        print(f"    {mname}: " +
              "  |  ".join(f"{s}: {counts.get(s, 0)}"
                           for s in ["OK", "FLAGGED", "HARD-PENALIZED"]))
    return df


# ============================================================================
# FIGURE 1 — Model comparison bar chart
# Train & test bars: mean +/- 1 SD across 50 per-fold scores
# ============================================================================
def plot_model_comparison(train_scores: dict, test_scores: dict):
    print("\n[Figure 1] Model comparison bar plots ...")
    metrics       = ["accuracy", "precision", "recall", "f1"]
    metric_labels = {"accuracy": "Accuracy", "precision": "Precision",
                     "recall": "Recall", "f1": "F1 Score"}

    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    axes = axes.flatten()
    x     = np.arange(len(ALL_MODELS))
    bar_w = 0.35

    for ax, metric in zip(axes, metrics):
        tr_means, tr_sds = [], []
        te_means, te_sds = [], []

        for mname in ALL_MODELS:
            tr = np.array(train_scores[mname][metric])
            te = np.array(test_scores[mname][metric])

            tr_means.append(np.mean(tr))
            tr_sds.append(np.std(tr, ddof=1))
            te_means.append(np.mean(te))
            te_sds.append(np.std(te, ddof=1))

        ax.bar(x - bar_w / 2, tr_means, bar_w,
               color="#B85042", alpha=0.85,
               yerr=tr_sds, capsize=4,
               error_kw=dict(elinewidth=1.4, ecolor="black", capthick=1.4),
               label="Train", zorder=3)

        ax.bar(x + bar_w / 2, te_means, bar_w,
               color=TEST_COLOR, alpha=0.80,
               yerr=te_sds, capsize=4,
               error_kw=dict(elinewidth=1.4, ecolor="black", capthick=1.4),
               label="Test", zorder=3)

        ax.set_xticks(x)
        ax.set_xticklabels(ALL_MODELS, rotation=20, ha="right", fontsize=10)
        ax.set_ylabel(metric_labels[metric], fontsize=11)
        ax.set_ylim(0, 1.15)
        ax.set_title(f"Model Performance: {metric_labels[metric]}", fontsize=12)
        ax.yaxis.grid(True, linestyle="--", alpha=0.5)
        ax.set_axisbelow(True)

        train_patch = mpatches.Patch(color="#B85042", alpha=0.85,
                                     label="Train (fold mean +/- 1 SD)")
        test_patch  = mpatches.Patch(color=TEST_COLOR, alpha=0.80,
                                     label="Test (fold mean +/- 1 SD)")
        ax.legend(handles=[train_patch, test_patch], fontsize=10, loc="upper left")

    n_folds = N_SPLITS * N_REPEATS
    plt.suptitle(
        f"Model Performance Comparison — {N_SPLITS}-Fold x {N_REPEATS}-Repeat CV "
        f"(n={n_folds} folds)  |  Train & Test mean +/- 1 SD\n"
        f"RF & XGB: hard-penalty gap diagnostic  "
        f"(primary ≤{GAP_PRIMARY:.0%}, fallback ≤{GAP_FALLBACK:.0%})",
        fontsize=13, y=1.01
    )
    plt.tight_layout()
    savefig(fig, "model_comparison/Fig1_model_comparison")
    plt.close(fig)


# ============================================================================
# FIGURE 2 — Confusion matrices
# Built from aggregated per-fold test predictions (fold-order concatenation)
# ============================================================================
def plot_confusion_matrices(agg_preds: dict, agg_true: dict):
    print("\n[Figure 2] Confusion matrices (aggregated per-fold test predictions) ...")
    class_labels = ["Low Risk", "Risky"]

    for mname in ALL_MODELS:
        cm  = confusion_matrix(agg_true[mname], agg_preds[mname])
        fig, ax = plt.subplots(figsize=(5, 4))
        _draw_cm(ax, cm, class_labels, mname)
        plt.tight_layout()
        safe = mname.replace(" ", "_")
        savefig(fig, f"confusion_matrix/Fig2_CM_{safe}")
        plt.draw(); plt.close(fig); plt.clf(); plt.cla()
        gc.collect()

    # Combined 2x3 figure
    fig, axes = plt.subplots(2, 3, figsize=(16, 10))
    axes = axes.flatten()
    for i, mname in enumerate(ALL_MODELS):
        cm = confusion_matrix(agg_true[mname], agg_preds[mname])
        _draw_cm(axes[i], cm, class_labels, mname)
    axes[-1].axis("off")
    plt.suptitle(
        f"Confusion Matrices — All Models  "
        f"(Aggregated per-fold test: {N_SPLITS}-Fold x {N_REPEATS}-Repeat CV)",
        fontsize=14, y=1.01
    )
    plt.tight_layout()
    savefig(fig, "confusion_matrix/Fig2_CM_combined")
    plt.draw(); plt.close(fig); plt.clf(); plt.cla()
    gc.collect()


def _draw_cm(ax, cm, labels, title):
    cmap = plt.cm.Blues
    ax.imshow(cm, interpolation="nearest", cmap=cmap)
    ax.set_title(f"Confusion Matrix\n{title}", fontsize=11, pad=10)
    tick_marks = np.arange(len(labels))
    ax.set_xticks(tick_marks);  ax.set_xticklabels(labels, fontsize=10)
    ax.set_yticks(tick_marks);  ax.set_yticklabels(labels, fontsize=10,
                                                    rotation=90, va="center")
    ax.set_xlabel("Predicted Class", fontsize=10)
    ax.set_ylabel("True Class", fontsize=10)
    thresh = cm.max() / 2.0
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            color = "white" if cm[i, j] > thresh else "black"
            ax.text(j, i, str(cm[i, j]),
                    ha="center", va="center",
                    fontsize=14, fontweight="bold", color=color)


# ============================================================================
# UNCERTAINTY INTERVAL HELPER
# Computes per-sample 5th and 95th percentile from per-repeat predictions.
# For each sample in the aggregated (fold-order) array, looks up its global
# index to retrieve all repeat-level probabilities stored in repeat_probs.
# Samples with fewer than N_REPEATS appearances still get a percentile from
# whatever repeats are available (minimum 1).
# ============================================================================
def compute_per_sample_uncertainty(agg_probs: dict, agg_dates: dict,
                                   repeat_probs: dict,
                                   X_len: int):
    """
    Returns for each model a dict with arrays aligned to the aggregated
    fold-order sequence:
        p5   : 5th  percentile of repeat predictions for that sample
        p95  : 95th percentile of repeat predictions for that sample
        lo   : agg_prob - p5   (lower error bar length)
        hi   : p95 - agg_prob  (upper error bar length)
    """
    uncertainty = {}
    for mname in ALL_MODELS:
        probs  = agg_probs[mname]          # length = n_agg (fold-concat)
        n_agg  = len(probs)

        # The aggregated array is built by concatenating test indices in fold
        # order.  We need to reconstruct which global sample index each
        # aggregated position corresponds to.  Because run_cv appends
        # dates_a[te_idx] we use the same logic: but dates may not be unique.
        # Instead we track the global indices directly via repeat_probs length.
        # We re-derive global-index order by matching: for each position in
        # the aggregated array find the global sample whose repeat_probs list
        # contains the corresponding probability value.
        #
        # Simpler robust approach: recompute fold membership using the same
        # RepeatedStratifiedKFold seed so we know te_idx per fold in order.
        # That lets us map aggregated position → global sample index exactly.

        p5_arr  = np.empty(n_agg)
        p95_arr = np.empty(n_agg)

        # Rebuild fold membership (same seed, same splitter as run_cv)
        # We need y to re-split; pass X_len as a proxy — we only need indices.
        # We import y from the outer scope via a closure trick: pass it in.
        # (Handled by caller passing y_arr into the helper — see call site.)
        # NOTE: y_arr is injected at call-site via the closure below.
        rskf = RepeatedStratifiedKFold(
            n_splits=N_SPLITS, n_repeats=N_REPEATS, random_state=RANDOM_STATE
        )

        pos = 0   # current position in the aggregated array
        for _, te_idx in rskf.split(np.zeros(X_len), _uncertainty_y_ref):
            for global_i in te_idx:
                repeats = repeat_probs[mname][global_i]
                if len(repeats) == 0:
                    # Fallback: use the aggregated value itself (no interval)
                    p5_arr[pos]  = probs[pos]
                    p95_arr[pos] = probs[pos]
                else:
                    p5_arr[pos]  = np.percentile(repeats,  5)
                    p95_arr[pos] = np.percentile(repeats, 95)
                pos += 1

        lo = np.clip(probs - p5_arr,  0, 1)
        hi = np.clip(p95_arr - probs, 0, 1)

        uncertainty[mname] = {"p5": p5_arr, "p95": p95_arr,
                               "lo": lo,    "hi": hi}
    return uncertainty


# Module-level variable injected before compute_per_sample_uncertainty runs
_uncertainty_y_ref = None


# ============================================================================
# FIGURE 3 — Marginal / uncertainty curve plots
# Uses aggregated per-fold test predicted probabilities (fold-order concat)
# ============================================================================
def plot_uncertainty(agg_probs: dict, agg_true: dict, agg_dates: dict,
                     repeat_probs: dict):
    print("\n[Figure 3] Marginal uncertainty plots (aggregated per-fold test) ...")

    # Pre-compute per-sample 5th/95th percentile intervals
    uncertainty = compute_per_sample_uncertainty(
        agg_probs, agg_dates, repeat_probs, len(_uncertainty_y_ref)
    )

    # Individual
    for mname in ALL_MODELS:
        probs  = agg_probs[mname]
        dates  = agg_dates[mname]
        y_true = agg_true[mname]
        x_idx  = np.arange(len(probs))
        unc    = uncertainty[mname]

        fig, ax = plt.subplots(figsize=(13, 5))
        _draw_uncertainty(ax, x_idx, probs, y_true, mname, dates, unc)
        plt.tight_layout()
        safe = mname.replace(" ", "_")
        savefig(fig, f"uncertainty/Fig3_Uncertainty_{safe}")
        plt.draw(); plt.close(fig); plt.clf(); plt.cla()
        gc.collect()

    # Combined
    fig, axes = plt.subplots(len(ALL_MODELS), 1,
                             figsize=(14, 5 * len(ALL_MODELS)),
                             sharex=False)
    for ax, mname in zip(axes, ALL_MODELS):
        probs  = agg_probs[mname]
        y_true = agg_true[mname]
        dates  = agg_dates[mname]
        x_idx  = np.arange(len(probs))
        unc    = uncertainty[mname]
        _draw_uncertainty(ax, x_idx, probs, y_true, mname, dates, unc)

    plt.suptitle(
        f"Predicted Probability — All Models  "
        f"(Aggregated per-fold test: {N_SPLITS}-Fold x {N_REPEATS}-Repeat CV)",
        fontsize=14, y=1.005
    )
    plt.tight_layout()
    savefig(fig, "uncertainty/Fig3_Uncertainty_combined")
    plt.draw(); plt.close(fig); plt.clf(); plt.cla()
    gc.collect()


def _draw_uncertainty(ax, x_idx, probs, y_true, mname, dates, unc):
    color      = COLORS[mname]
    band_color = BAND_PALETTE[mname]

    y_pred  = (probs >= DECISION_THR).astype(int)
    correct = y_pred == y_true

    # ── Shaded 5%–95% band (sorted by x_idx for a smooth fill) ───────────────
    sort_order = np.argsort(x_idx)
    xs         = x_idx[sort_order]
    p5s        = unc["p5"][sort_order]
    p95s       = unc["p95"][sort_order]
    ax.fill_between(xs, p5s, p95s,
                    color=band_color, alpha=0.25, zorder=1,
                    label="5%–95% uncertainty band")

    # ── Scatter dots with vertical error bars ────────────────────────────────
    # Correct predictions
    ax.errorbar(
        x_idx[correct], probs[correct],
        yerr=[unc["lo"][correct], unc["hi"][correct]],
        fmt="o", color=color, markersize=4, alpha=0.70,
        ecolor=band_color, elinewidth=0.8, capsize=2, capthick=0.8,
        zorder=5, label="Correct prediction"
    )
    # Incorrect predictions
    ax.errorbar(
        x_idx[~correct], probs[~correct],
        yerr=[unc["lo"][~correct], unc["hi"][~correct]],
        fmt="x", color="red", markersize=5, alpha=0.65,
        ecolor="#FFAAAA", elinewidth=0.8, capsize=2, capthick=0.8,
        zorder=6, label="Incorrect prediction"
    )

    ax.axhline(DECISION_THR, color="black", linestyle="--",
               linewidth=1.5, label="Decision Threshold (0.5)")
    ax.set_ylim(-0.05, 1.10)
    ax.set_title(
        f"{mname} — Predicted Probability (aggregated per-fold test)", fontsize=11)
    ax.set_ylabel("Probability of Risky Events", fontsize=10)
    ax.set_xlabel("Aggregated Test Sample Index (fold order)", fontsize=10)
    ax.yaxis.grid(True, linestyle="--", alpha=0.4)
    ax.set_axisbelow(True)
    ax.legend(fontsize=9, loc="upper right")


# ============================================================================
# FIGURE 4 — Probability density plots
# Uses aggregated per-fold test predicted probabilities (fold-order concat)
# ============================================================================
def plot_density(agg_probs: dict, agg_true: dict):
    print("\n[Figure 4] Probability density plots (aggregated per-fold test) ...")

    # Individual: class-conditional KDE (Low Risk vs Risky)
    for mname in ALL_MODELS:
        probs  = agg_probs[mname]
        y_true = agg_true[mname]
        fig, ax = plt.subplots(figsize=(8, 5))
        _draw_density(ax, probs, y_true, mname)
        plt.tight_layout()
        safe = mname.replace(" ", "_")
        savefig(fig, f"density/Fig4_Density_{safe}")
        plt.draw(); plt.close(fig); plt.clf(); plt.cla()
        gc.collect()

    # Combined: overall KDE for all models on one axis
    fig, ax = plt.subplots(figsize=(9, 6))
    for mname in ALL_MODELS:
        probs = agg_probs[mname].copy()
        color = COLORS[mname]
        if np.std(probs) < 1e-8:
            probs = probs + np.random.normal(0, 1e-6, len(probs))
        kde    = gaussian_kde(probs, bw_method=0.15)
        x_grid = np.linspace(0, 1, 400)
        ax.plot(x_grid, kde(x_grid), color=color, linewidth=2, label=mname)
        ax.fill_between(x_grid, kde(x_grid), alpha=0.12, color=color)

    ax.axvline(DECISION_THR, color="black", linestyle="--",
               linewidth=1.5, label="Decision Threshold (0.5)")
    ax.set_xlabel("Predicted Probability of Risky Events", fontsize=11)
    ax.set_ylabel("Density", fontsize=11)
    ax.set_title(
        f"Probability Distribution — All Models  "
        f"({N_SPLITS}-Fold x {N_REPEATS}-Repeat CV, aggregated per-fold test)",
        fontsize=12
    )
    ax.legend(fontsize=10)
    ax.yaxis.grid(True, linestyle="--", alpha=0.4)
    ax.set_axisbelow(True)
    plt.tight_layout()
    savefig(fig, "density/Fig4_Density_combined")
    plt.draw(); plt.close(fig); plt.clf(); plt.cla()
    gc.collect()


def _draw_density(ax, probs, y_true, mname):
    color  = COLORS[mname]
    x_grid = np.linspace(0, 1, 400)

    for cls, cls_label, ls, alpha in [(0, "Low Risk",  "-",  0.40),
                                       (1, "Risky",     "--", 0.35)]:
        sub = probs[y_true == cls].copy()
        if len(sub) < 3:
            continue
        if np.std(sub) < 1e-8:
            sub = sub + np.random.normal(0, 1e-6, len(sub))
        kde = gaussian_kde(sub, bw_method=0.15)
        ax.plot(x_grid, kde(x_grid), color=color, linewidth=2,
                linestyle=ls, label=f"{cls_label} (true)")
        ax.fill_between(x_grid, kde(x_grid), alpha=alpha, color=color)

    ax.axvline(DECISION_THR, color="black", linestyle="--",
               linewidth=1.5, label="Decision Threshold (0.5)")
    ax.set_xlabel("Predicted Probability", fontsize=11)
    ax.set_ylabel("Density", fontsize=11)
    ax.set_title(f"Probability Density — {mname}", fontsize=11)
    ax.legend(fontsize=10)
    ax.yaxis.grid(True, linestyle="--", alpha=0.4)
    ax.set_axisbelow(True)


# ============================================================================
# FIGURE 5 — Feature Importance (intrinsic on full dataset, both sides)
# ============================================================================
def get_feature_importances(X, y):
    print("\n[FI] Computing feature importances on full dataset ...")

    X_np = X.values if isinstance(X, pd.DataFrame) else X

    scaler_full = StandardScaler()
    X_full_sc   = scaler_full.fit_transform(X_np)
    X_full_df   = pd.DataFrame(X_full_sc, columns=FEATURES)

    fi_train, fi_test, fi_train_std, fi_test_std, method_label = {}, {}, {}, {}, {}

    builders = {
        "Logistic Regression": build_lr,
        "Decision Tree":       build_dt,
        "XGBoost":             build_xgb,
        "Random Forest":       build_rf,
    }

    for mname, builder in builders.items():
        print(f"  Feature importance: {mname} ...")
        model = builder()
        model.fit(X_full_df, y)

        if mname == "Logistic Regression":
            raw = np.abs(model.coef_[0])
            imp = raw / (raw.sum() + 1e-12)
            std = np.zeros(len(FEATURES))
            method_label[mname] = "|Coefficient| (normalised)"

        elif mname in ("Decision Tree", "Random Forest"):
            imp = model.feature_importances_
            std = np.zeros(len(FEATURES))
            method_label[mname] = "Built-in Impurity Importance"

        elif mname == "XGBoost":
            imp = model.feature_importances_
            std = np.zeros(len(FEATURES))
            method_label[mname] = "Built-in Gain Importance"

        fi_train[mname]     = imp
        fi_test[mname]      = imp
        fi_train_std[mname] = std
        fi_test_std[mname]  = std

    # ── ANN — permutation importance on full dataset (accuracy scoring) ───────
    print("  Feature importance: ANN ...")
    from sklearn.inspection import permutation_importance as perm_imp

    ann = ANNWrapper(X_full_sc.shape[1])
    ann.fit(X_full_sc, y)

    pi = perm_imp(
        ann, X_full_sc, y,
        n_repeats=PERM_N_REPEATS,
        random_state=RANDOM_STATE,
        scoring="accuracy",
    )
    fi_train["ANN"]     = pi.importances_mean
    fi_test["ANN"]      = pi.importances_mean
    fi_train_std["ANN"] = pi.importances_std
    fi_test_std["ANN"]  = pi.importances_std
    method_label["ANN"] = "Permutation Importance (Accuracy, full dataset)"

    keras.backend.clear_session()
    gc.collect()

    return fi_train, fi_test, fi_train_std, fi_test_std, method_label


def _draw_fi_bar(ax, importances, stds, title, color, split_label,
                 method_lbl, feature_names):
    """Horizontal bar chart sorted by importance descending."""
    idx   = np.argsort(importances)
    vals  = importances[idx]
    errs  = stds[idx]
    feats = [feature_names[i] for i in idx]
    y_pos = np.arange(len(feats))

    bars = ax.barh(y_pos, vals, xerr=errs,
                   color=color, alpha=0.82,
                   error_kw=dict(elinewidth=1.2, ecolor="black", capsize=3),
                   height=0.65)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(feats, fontsize=10)
    ax.set_xlabel("Importance Score", fontsize=10)
    ax.set_title(f"{title}\n({split_label} | {method_lbl})", fontsize=10)
    ax.xaxis.grid(True, linestyle="--", alpha=0.4)
    ax.set_axisbelow(True)
    mx = max(abs(vals)) if len(vals) > 0 else 1e-12
    for bar, v in zip(bars, vals):
        ax.text(bar.get_width() + mx * 0.02,
                bar.get_y() + bar.get_height() / 2,
                f"{v:.3f}", va="center", ha="left", fontsize=8)


def plot_feature_importance(fi_train, fi_test,
                             fi_train_std, fi_test_std,
                             method_label):
    print("\n[Figure 5] Feature importance plots ...")

    # Individual: one figure per model (train left, test right)
    for mname in ALL_MODELS:
        color = COLORS[mname]
        fig, axes = plt.subplots(1, 2, figsize=(14, 6))
        _draw_fi_bar(axes[0], fi_train[mname], fi_train_std[mname],
                     mname, color, "Full Dataset (train)",
                     method_label[mname], FEATURES)
        _draw_fi_bar(axes[1], fi_test[mname], fi_test_std[mname],
                     mname, color, "Full Dataset (test view)",
                     method_label[mname], FEATURES)
        plt.suptitle(f"Feature Importance — {mname}", fontsize=13, y=1.02)
        plt.tight_layout()
        safe = mname.replace(" ", "_")
        savefig(fig, f"feature_importance/Fig5_FI_{safe}")
        plt.draw(); plt.close(fig); plt.clf(); plt.cla()
        gc.collect()

    # Combined: all models in one figure
    fig, axes = plt.subplots(len(ALL_MODELS), 2,
                             figsize=(16, 5 * len(ALL_MODELS)))
    for row, mname in enumerate(ALL_MODELS):
        color = COLORS[mname]
        _draw_fi_bar(axes[row, 0], fi_train[mname], fi_train_std[mname],
                     mname, color, "Full Dataset (train)",
                     method_label[mname], FEATURES)
        _draw_fi_bar(axes[row, 1], fi_test[mname], fi_test_std[mname],
                     mname, color, "Full Dataset (test view)",
                     method_label[mname], FEATURES)
    plt.suptitle("Feature Importance — All Models  (Intrinsic, Full Dataset)",
                 fontsize=14, y=1.005)
    plt.tight_layout()
    savefig(fig, "feature_importance/Fig5_FI_combined")
    plt.draw(); plt.close(fig); plt.clf(); plt.cla()
    gc.collect()

    # Overlay: all models grouped by feature
    fig, axes = plt.subplots(1, 2, figsize=(16, 7))
    for split_idx, (fi_dict, std_dict, split_lbl) in enumerate([
            (fi_train, fi_train_std, "Full Dataset (train)"),
            (fi_test,  fi_test_std,  "Full Dataset (test view)")]):
        ax      = axes[split_idx]
        xp      = np.arange(len(FEATURES))
        n_m     = len(ALL_MODELS)
        total_w = 0.8
        bar_w   = total_w / n_m
        offsets = np.linspace(-total_w / 2 + bar_w / 2,
                               total_w / 2 - bar_w / 2, n_m)
        for i, mname in enumerate(ALL_MODELS):
            ax.bar(xp + offsets[i], fi_dict[mname], bar_w,
                   color=COLORS[mname], alpha=0.82,
                   yerr=std_dict[mname],
                   error_kw=dict(elinewidth=0.8, ecolor="black", capsize=2),
                   label=mname)
        ax.set_xticks(xp)
        ax.set_xticklabels(FEATURES, rotation=30, ha="right", fontsize=9)
        ax.set_ylabel("Importance Score", fontsize=10)
        ax.set_title(f"Feature Importance — {split_lbl}", fontsize=11)
        ax.yaxis.grid(True, linestyle="--", alpha=0.4)
        ax.set_axisbelow(True)
        ax.legend(fontsize=9, loc="upper right")
    plt.suptitle("Feature Importance Comparison — All Models", fontsize=13, y=1.02)
    plt.tight_layout()
    savefig(fig, "feature_importance/Fig5_FI_overlay")
    plt.draw(); plt.close(fig); plt.clf(); plt.cla()
    gc.collect()

    # CSV
    rows = []
    for mname in ALL_MODELS:
        for feat, tr_v, tr_s, te_v, te_s in zip(
                FEATURES,
                fi_train[mname], fi_train_std[mname],
                fi_test[mname],  fi_test_std[mname]):
            rows.append({
                "Model":            mname,
                "Feature":          feat,
                "Train_Importance": tr_v,
                "Train_Std":        tr_s,
                "Test_Importance":  te_v,
                "Test_Std":         te_s,
                "Method":           method_label[mname],
            })
    pd.DataFrame(rows).to_csv(
        os.path.join(OUTPUT_DIR, "tables", "feature_importance.csv"), index=False)
    print(f"  Saved → {OUTPUT_DIR}/tables/feature_importance.csv")


# ============================================================================
# METRICS SUMMARY TABLE
# Includes HARD-PENALIZED fold counts for RF and XGB as extra columns.
# ============================================================================
def save_metrics_table(train_scores: dict, test_scores: dict,
                       penalty_df: pd.DataFrame):
    rows = []
    for mname in ALL_MODELS:
        row = {"Model": mname}
        for met in ["accuracy", "precision", "recall", "f1"]:
            tr = np.array(train_scores[mname][met])
            te = np.array(test_scores[mname][met])
            row[f"Train_CV_{met}_mean"] = np.mean(tr)
            row[f"Train_CV_{met}_std"]  = np.std(tr, ddof=1)
            row[f"Test_CV_{met}_mean"]  = np.mean(te)
            row[f"Test_CV_{met}_std"]   = np.std(te, ddof=1)

        # Annotate HARD-PENALIZED fold counts for RF and XGB
        if mname in PENALIZED_MODELS and len(penalty_df) > 0:
            sub = penalty_df[penalty_df["Model"] == mname]
            hp_count  = int((sub["Gap_Status"] == "HARD-PENALIZED").sum())
            flg_count = int((sub["Gap_Status"] == "FLAGGED").sum())
            ok_count  = int((sub["Gap_Status"] == "OK").sum())
            row["Gap_OK_folds"]             = ok_count
            row["Gap_FLAGGED_folds"]        = flg_count
            row["Gap_HARD_PENALIZED_folds"] = hp_count
            row["Gap_note"] = (
                f"{hp_count} HARD-PENALIZED fold(s) kept in aggregation "
                f"(Option B — predictions and scores retained)"
                if hp_count > 0 else ""
            )
        else:
            row["Gap_OK_folds"]             = ""
            row["Gap_FLAGGED_folds"]        = ""
            row["Gap_HARD_PENALIZED_folds"] = ""
            row["Gap_note"]                 = ""

        rows.append(row)

    path = os.path.join(OUTPUT_DIR, "tables", "metrics_summary.csv")
    pd.DataFrame(rows).to_csv(path, index=False)
    print(f"  Saved → {path}")


# ============================================================================
# MAIN
# ============================================================================
def main():
    print("\nTensorFlow version:", tf.__version__)
    print("Matplotlib backend:", matplotlib.get_backend())
    set_global_seed(RANDOM_STATE)
    setup_output_dirs()

    print("\nCurrent working directory:", os.getcwd())
    print("Absolute output directory: ", os.path.abspath(OUTPUT_DIR))

    # -- Load data -------------------------------------------------------------
    X, y, dates, df = load_data()

    # -- Run full CV loop ------------------------------------------------------
    print(f"\n[CV] Running {N_SPLITS}-fold x {N_REPEATS}-repeat stratified CV "
          f"({N_SPLITS * N_REPEATS} total folds) ...")
    print(f"     RF & XGB: hard-penalty gap diagnostic  "
          f"(primary ≤{GAP_PRIMARY:.0%}, fallback ≤{GAP_FALLBACK:.0%})")

    # Expose y to the uncertainty helper (needed to reconstruct fold membership)
    global _uncertainty_y_ref
    _uncertainty_y_ref = y
    (train_scores, test_scores,
     agg_preds, agg_probs, agg_true, agg_dates,
     penalty_log, repeat_probs) = run_cv(X, y, dates)

    # -- Penalty log CSV -------------------------------------------------------
    print("\n[Tables] Saving gap penalty log ...")
    penalty_df = save_penalty_log(penalty_log)

    # -- Metrics table ---------------------------------------------------------
    print("\n[Tables] Saving metrics summary ...")
    save_metrics_table(train_scores, test_scores, penalty_df)

    # -- FIGURE 1 — Bar plots with SD error bars --------------------------------
    plot_model_comparison(train_scores, test_scores)

    # -- FIGURE 2 — Confusion matrices -----------------------------------------
    plot_confusion_matrices(agg_preds, agg_true)

    # -- FIGURE 3 — Marginal / uncertainty curves ------------------------------
    plot_uncertainty(agg_probs, agg_true, agg_dates, repeat_probs)

    # -- FIGURE 4 — Density plots ----------------------------------------------
    plot_density(agg_probs, agg_true)

    # -- FIGURE 5 — Feature importance -----------------------------------------
    fi_train, fi_test, fi_train_std, fi_test_std, method_label = \
        get_feature_importances(X, y)
    plot_feature_importance(fi_train, fi_test, fi_train_std, fi_test_std, method_label)

    print("\n[OK]  All figures saved to:", os.path.join(OUTPUT_DIR, "figures"))
    print("   Metrics table    saved to:", os.path.join(OUTPUT_DIR, "tables",
                                                         "metrics_summary.csv"))
    print("   Feature imp CSV  saved to:", os.path.join(OUTPUT_DIR, "tables",
                                                         "feature_importance.csv"))
    print("   Gap penalty log  saved to:", os.path.join(OUTPUT_DIR, "tables",
                                                         "gap_penalty_log.csv"))


if __name__ == "__main__":
    main()
